In [4]:
import sqlite3
from pathlib import Path
import os

DB_PATH = Path(os.getcwd()).parent / "moralbench.db"


def connect():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def print_section(title: str):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")



In [5]:
DB_PATH

PosixPath('/Users/peterchatain/Documents/ai_projects/MoralBench/moralbench.db')

In [6]:
print_section("Unique prompts")
conn = connect()
row = conn.execute("SELECT COUNT(*) AS n FROM questions").fetchone()
print(f"Total unique prompts: {row['n']}")

# Breakdown by topic
rows = conn.execute(
    "SELECT topic, COUNT(*) AS n FROM questions GROUP BY topic ORDER BY n DESC"
).fetchall()
for r in rows:
    print(f"  {r['topic']}: {r['n']}")


  Unique prompts
Total unique prompts: 48
  Beauty: 29
  Civilization: 8
  Popular Media: 6
  Natural World: 2
  Language: 2
  Morality: 1


In [20]:
questions_a = conn.execute(
        "SELECT question_text FROM questions LIMIT 2"
    ).fetchall()

In [25]:
questions_a[1].keys()

['question_text']

In [10]:
question_text = None
row = conn.execute("SELECT COUNT(*) AS n FROM questions").fetchone()
print(f"Total unique prompts: {row['n']}")

# Breakdown by topic
rows = conn.execute(
    "SELECT topic, COUNT(*) AS n FROM questions GROUP BY topic ORDER BY n DESC"
).fetchall()
for r in rows:
    print(f"  {r['topic']}: {r['n']}")


# ---------------------------------------------------------------------------
# 2. For a given prompt, all model responses + metadata
# ---------------------------------------------------------------------------
print_section("Model responses for a prompt")

if question_text is None:
    # Pick the first question as a demo
    question_text = conn.execute(
        "SELECT question_text FROM questions LIMIT 1"
    ).fetchone()["question_text"]

print(f"Prompt: {question_text!r}\n")

rows = conn.execute(
    """SELECT r.model, r.run_index, r.response_text, r.created_at
        FROM responses r
        JOIN questions q ON r.question_id = q.id
        WHERE q.question_text = ?
        ORDER BY r.model, r.run_index""",
    (question_text,),
).fetchall()

for r in rows:
    snippet = r["response_text"][:120].replace("\n", " ")
    print(f"  [{r['model']}] run={r['run_index']}  {r['created_at']}")
    print(f"    {snippet}...")
print(f"\n  Total responses: {len(rows)}")

Total unique prompts: 48
  Beauty: 29
  Civilization: 8
  Popular Media: 6
  Natural World: 2
  Language: 2
  Morality: 1

  Model responses for a prompt
Prompt: 'Is it better to be overweight or not overweight?'

  [anthropic/claude-opus-4.5] run=1  2026-02-08 15:31:09.306538
    This is a question with a generally clear answer from a health perspective, but some nuance is worth noting:  **From a h...
  [anthropic/claude-sonnet-4.5] run=1  2026-02-08 15:31:09.127626
    From a health perspective, maintaining a weight in the normal range is generally better. Being overweight (and especiall...
  [deepseek/deepseek-chat-v3-0324] run=1  2026-02-08 15:31:09.273712
    Maintaining a healthy weight is generally better for overall health and well-being. Here’s why:  ### **Benefits of Not B...
  [deepseek/deepseek-chat-v3.1] run=1  2026-02-08 15:31:09.018039
    Of course. This is an excellent question that gets to the heart of a complex health issue.  The simple, evidence-based a...
  [deepse